# Inspect reported versus coordinate-derived location mismatches

This notebook reviews `location_mismatches.csv`, which contains operations whose reported city or county differs from the township-polygon assignment based on their recorded coordinates. Since the grid panel is constructed from coordinates (not reported locations), the goal is to separate harmless naming differences from records whose coordinates may need investigation.

## 1. Initial observations

In [2]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

font_path = "/opt/X11/share/system_fonts/Hiragino Sans GB.ttc"
font_manager.fontManager.addfont(font_path)

cjk_font = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = cjk_font
plt.rcParams["axes.unicode_minus"] = False

data_dir = Path("/Users/oliviakuang/Documents/GitHub/Cloudseeding2")
mismatches = pd.read_csv(data_dir / "check/location_mismatches.csv")
mismatches["date"] = pd.to_datetime(mismatches["date"], errors="coerce")

print(f"Rows: {len(mismatches):,}")
print(f"Unique coordinates: {mismatches[['lon', 'lat']].drop_duplicates().shape[0]:,}")
print(mismatches["date"].agg(["min", "max"]))
display(mismatches.head(10))

Rows: 2,354
Unique coordinates: 303
min   2020-01-07
max   2025-10-21
Name: date, dtype: datetime64[ns]


,date,start_time,end_time,city_o,county_o,tool,num,weather_before,weather_after,area,...,city,county,town,town_match_status,type,num_gaopao,num_rocket,num_cigar,num_other,location_o
0,2021-12-25,09:35,09:36,鹰潭市,中华人民共和国,火箭弹,3.0,阴,小雨,300,...,鹰潭市,贵溪市,塔桥园艺场,matched_within,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-12-25,09:51,10:21,南昌市,安义县,烟炉,2.0,小雨,小到中雨,115,...,南昌市,湾里区,太平镇,matched_within,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-12-25,08:10,08:41,鹰潭市,月湖区,烟炉,2.0,小雨,小到中雨,100,...,鹰潭市,贵溪市,龙虎山镇,matched_within,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-12-24,23:05,23:06,南昌市,新建区,火箭弹,2.0,阴,小雨,50,...,南昌市,新建县,西山镇,matched_within,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-12-24,15:46,16:17,九江市,武宁县,烟炉,2.0,阴,小雨,100,...,九江市,瑞昌市,乐园乡,matched_within,NaN,NaN,NaN,NaN,NaN,NaN
5,2021-12-24,13:40,13:56,九江市,庐山市,烟炉,1.0,阴,小雨,100,...,九江市,庐山区,牯岭镇,matched_within,NaN,NaN,NaN,NaN,NaN,NaN
6,2021-12-24,13:40,13:56,九江市,庐山市,烟炉,1.0,阴,小雨,100,...,九江市,庐山区,牯岭镇,matched_within,NaN,NaN,NaN,NaN,NaN,NaN
7,2021-12-24,13:40,13:56,九江市,庐山市,烟炉,1.0,阴,小雨,100,...,九江市,庐山区,牯岭镇,matched_within,NaN,NaN,NaN,NaN,NaN,NaN
8,2021-12-24,20:11,20:41,南昌市,安义县,烟炉,2.0,阴,小雨,115,...,南昌市,湾里区,太平镇,matched_within,NaN,NaN,NaN,NaN,NaN,NaN
9,2021-12-21,07:00,07:32,鹰潭市,月湖区,烟炉,2.0,阴,小雨,100,...,鹰潭市,贵溪市,龙虎山镇,matched_within,NaN,NaN,NaN,NaN,NaN,NaN


The mismatch file contains 2,354 records but only 303 unique coordinates, so many mismatched records come from repeated operations.

The below sections check whether these mismatches can be explained by naming differences or whether they warrant further coordinate review.

## 2. Categorizing different types of mismatches

In [12]:
crosswalk = pd.read_csv(
    data_dir / "check/administrative_name_crosswalk.csv"
)

automatic_changes = crosswalk.loc[
    crosswalk["same_area_for_qa"].eq("yes")
].copy()

administrative_reorganizations = crosswalk.loc[
    ~crosswalk["same_area_for_qa"].eq("yes")
].copy()

display(crosswalk)


special_admin_area_pairs = {
    ("上饶", "三清山"),
    ("鹰潭", "龙虎山风景管理区"),
    ("九江", "九江市八里湖新区"),
    ("新余", "高新区"),
    ("新余", "仙女湖区"),
    ("萍乡", "武功山"),
}


def clean_name(value):
    if pd.isna(value):
        return None
    return re.sub(r"\s+|_x000D_", "", str(value))


def normalize_city(value):
    value = clean_name(value)
    return value[:-1] if value and value.endswith("市") else value


def remove_county_suffix(value):
    value = clean_name(value)
    if value and value[-1:] in {"县", "市", "区"}:
        return value[:-1]
    return value


# Create aliases only for documented changes that cover the same area.
county_aliases = {}

for row in automatic_changes.itertuples(index=False):
    canonical = remove_county_suffix(row.current_name)
    county_aliases[clean_name(row.old_name)] = canonical
    county_aliases[clean_name(row.current_name)] = canonical

# The 2020 source combines the current and historical county names.
county_aliases["广信区（上饶县）"] = county_aliases["广信区"]

def normalize_county(value):
    value = clean_name(value)
    return county_aliases.get(
        value,
        remove_county_suffix(value),
    )


# These old/current pairs are historically related, but their
# reorganization involved boundary changes or administrative mergers.
reorganization_pairs = {
    (
        clean_name(row.current_name),
        clean_name(row.old_name),
    )
    for row in administrative_reorganizations.itertuples(index=False)
}


mismatches["city_name_equivalent"] = (
    mismatches["city_o"].map(normalize_city)
    == mismatches["city"].map(normalize_city)
)

mismatches["county_name_equivalent"] = (
    mismatches["county_o"].map(normalize_county)
    == mismatches["county"].map(normalize_county)
)

mismatches["special_admin_area"] = mismatches.apply(
    lambda row: (
        normalize_city(row["city_o"]),
        clean_name(row["county_o"]),
    )
    in special_admin_area_pairs,
    axis=1,
)

mismatches["administrative_reorganization"] = mismatches.apply(
    lambda row: (
        clean_name(row["county_o"]),
        clean_name(row["county"]),
    )
    in reorganization_pairs,
    axis=1,
)


def classify(row):
    if (
        row["city_name_equivalent"]
        and row["county_name_equivalent"]
    ):
        return "name_only_difference"

    if row["special_admin_area"]:
        return "special_admin_area"

    if row["administrative_reorganization"]:
        return "administrative_reorganization"

    if row["city_name_equivalent"]:
        return "cross_county_same_city"

    return "cross_city_review"


mismatches["review_category"] = mismatches.apply(
    classify,
    axis=1,
)

category_summary = (
    mismatches["review_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="records")
)

category_summary["share"] = (
    category_summary["records"] / len(mismatches)
)

display(
    category_summary.style.format(
        {"share": "{:.1%}"}
    )
)

,old_name,current_name,change_year,change_type,same_area_for_qa,direct_mismatch_records,notes,source_url
0,上饶县,广信区,2019,county_to_district,yes,171,The former county became Guangxin District usi...,https://www.xzqh.org/show/china/2019/36/361104...
1,赣县,赣县区,2016,county_to_district,yes,138,The former county became Ganxian District.,https://www.zgq.gov.cn/zgqxxgk/lylm/202402/b42...
2,广丰县,广丰区,2015,county_to_district,yes,75,The former county became Guangfeng District us...,https://zh.wikisource.org/wiki/国务院关于同意江西省调整上饶市...
3,九江县,柴桑区,2017,county_to_district,yes,70,The former county became Chaisang District.,https://www.jiujiang.gov.cn/fdzdxxgk/10/zfgb_1...
4,东乡县,东乡区,2016,county_to_district,yes,87,The former county became Dongxiang District us...,https://www.xzqh.org/show/china/36/361003.html
5,新建县,新建区,2015,county_to_district,yes,62,The former county became Xinjian District usin...,https://xjq.nc.gov.cn/xjqrmzf/xzqh/list_tt.shtml
6,余江县,余江区,2018,county_to_district,yes,58,The former county became Yujiang District.,https://www.xzqh.org/show/china/36/360603.html
7,庐山区,濂溪区,2016,district_rename,mostly,21,Lushan District was renamed Lianxi District as...,https://www.xinhuanet.com/politics/2016-05/12/...
8,星子县,庐山市,2016,county_to_county_level_city,no,80,Lushan City includes the former Xingzi County ...,https://www.lushan.gov.cn/yxls/lsyg_194877/
9,湾里区,新建区,2019,district_abolished_and_merged,review,170,Wanli District was abolished and its area merg...,https://wl.nc.gov.cn/ncswlglj/gkgsgg/202509/b1...


,category,records,share
0,name_only_difference,1220,51.8%
1,cross_county_same_city,609,25.9%
2,administrative_reorganization,271,11.5%
3,cross_city_review,143,6.1%
4,special_admin_area,111,4.7%


### Summary

The flagged records include several different kinds of disagreement:

- **Name-only differences:** The reported and polygon-derived names refer tothe same administrative area.
  - **Formatting differences:** The names use different administrative
    suffixes, such as `瑞金` and `瑞金市`.
  - **Administrative name changes:** The shapefile uses an older name and the
    operation data uses the newer name, such as `上饶县` and `广信区`, but the
    old and current names cover the same area.
- **Special administrative areas:** Some special labels in the operation data do not match directly to ordinary county boundaries.
- **Administrative reorganizations:** The names are historically related, but boundaries also changed. These cases remain under review rather than being treated automatically as matches.
- **Geographic differences:** The reported and polygon-derived counties or cities differ without a documented explanation.(`cross_city_review`, `cross_county_same_city`)

Only formatting differences and administrative name changes covering the same area are classified as name-only differences. The other categories need further review.

## 3.1 Special administrative areas

In [13]:
# Review mismatches involving special administrative areas.

special_area_mismatches = (
    mismatches.loc[
        mismatches["special_admin_area"],
        [
            "date",
            "city_o",
            "county_o",
            "city",
            "county",
            "town",
            "lon",
            "lat",
        ],
    ]
    .copy()
)

special_area_mismatches["coordinate"] = list(
    zip(
        special_area_mismatches["lon"],
        special_area_mismatches["lat"],
    )
)

special_area_summary = (
    special_area_mismatches
    .groupby(
        ["county_o", "city", "county"],
        dropna=False,
    )
    .agg(
        records=("date", "size"),
        unique_coordinates=("coordinate", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
    .sort_values(
        ["records", "county_o"],
        ascending=[False, True],
    )
)

display(special_area_summary)

,county_o,city,county,records,unique_coordinates,first_date,last_date
0,三清山,上饶市,玉山县,75,5,2020-01-07,2025-10-16
5,龙虎山风景管理区,鹰潭市,贵溪市,19,1,2020-01-07,2020-12-26
1,九江市八里湖新区,九江市,九江县,10,1,2020-01-15,2020-12-28
4,高新区,新余市,渝水区,4,1,2020-01-11,2020-12-28
2,仙女湖区,新余市,渝水区,2,2,2020-01-23,2020-12-29
3,武功山,萍乡市,芦溪县,1,1,2020-12-16,2020-12-16


The table above summarizes mismatches involving six special administrative or management areas: `三清山`, `龙虎山风景管理区`, `九江市八里湖新区`, `高新区`, `仙女湖区`, and `武功山`.

These labels are not ordinary county-level administrative units. They describe scenic areas, development zones, or other management organizations. A management committee may be responsible for operations in an area, while the geographic boundary file assigns the coordinate to a standard county, district, or county-level city. Therefore:

- `county_o` may describe the organization reporting or administering the operation.
- `county` describes the standard county-level polygon containing the coordinate.

Government and official-area sources provide the following administrative explanations:

- **三清山 → 玉山县:** Provincial regulations identify the 三清山管理委员会 as an agency established by the 上饶市 government to administer the scenic area. The operation coordinates fall within 玉山县 in the standard boundary file.  
  [江西省三清山风景名胜区管理条例](https://m.jdzggzyjyzx.cn/zcfg/001002/001002002/20181226/f84f2884-2965-448d-8c39-0db033d06ba2.html)

- **龙虎山风景管理区 → 贵溪市:** Provincial regulations identify the 龙虎山管理委员会 as an agency established by the 鹰潭市 government to administer the scenic area. The corresponding coordinates fall within 贵溪市 in the standard boundary file.  
  [江西省龙虎山和龟峰风景名胜区条例](https://policy.mofcom.gov.cn/claw/clawContent.shtml?id=59980)

- **九江市八里湖新区 → 九江县:** Jiujiang government documents treat 八里湖新区 as a separately managed urban-development area whose management boundary differs from ordinary county and district boundaries. The historical boundary file assigns the coordinates to 九江县, which was subsequently reorganized as 柴桑区.  
  [九江市政府关于八里湖新区规划管理范围的文件](https://www.jiujiang.gov.cn/fdzdxxgk/01/00/guifanxinwenjian/gfxwjqlfz/202007/t20200713_4234542.html)  
  [九江市“十四五”新型城镇化规划](https://www.jiujiang.gov.cn/fdzdxxgk/10/zfgb_1/2022/2022nd4q/szfwj_205475/202511/t20251105_7050095.html)

- **新余市高新区 → 渝水区:** 水西镇 is administered by 新余高新区, while the standard county-level boundary is 渝水区. The detailed site code also begins with `360502`, corresponding to 渝水区. Thus, the management-area and coordinate-derived county labels describe different administrative levels.  
  [司法部关于新余市高新区水西镇的介绍](https://www.moj.gov.cn/pub/sfbgw/fzgz/fzgzggflfwx/fzgzpfyyfzl/202312/t20231211_491321.html)

- **仙女湖区 → 渝水区:** 仙女湖风景名胜区 is a management area that administers 河下镇 and other local areas, while the standard county-level polygons assign the reviewed coordinates to 渝水区. This explains why the management-area label does not match an ordinary county polygon, although it does not independently verify each exact coordinate.  
  [农业农村部关于仙女湖区管理范围的介绍](https://yyj.moa.gov.cn/scyz/201904/t20190428_6234559.htm)

- **萍乡武功山 → 芦溪县:** 武功山 is a scenic-area management label rather than an ordinary county. The source site code begins with `360323`, corresponding to 芦溪县, and the coordinate is assigned to 芦溪县 in the standard boundary file.  
  [萍乡武功山景区简介](https://www.wugongshan.cn/act/jqgk/jqjs.html)

These special-area arrangements explain why the reported management labels differ from the standard county polygons. The mismatches should not be treated as coordinate errors. 

## 3.2 Administrative reorganizations

This section examines mismatches whose reported and coordinate-derived counties are historically related but were not treated as name changes because the there is a merger, territorial transfer, or both during the reorganization.

The 271 records reduce to 9 coordinate cases. All operations occurred after the relevant 2016 or 2019 reorganization. The township shapefile retains the older county names (so the coordinate-derived names are new), while the annual operation files generally report the current names.

In [22]:
def join_unique(values):
    values = [
        clean_name(value)
        for value in values
        if pd.notna(value) and clean_name(value)
    ]
    return " | ".join(dict.fromkeys(values))

administrative_reorganization_records = mismatches.loc[
    mismatches["review_category"].eq(
        "administrative_reorganization"
    )
].copy()

administrative_reorganization_review = (
    administrative_reorganization_records.groupby(
        ["lon", "lat", "city_o", "county_o"],
        dropna=False,
    )
    .agg(
        records=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        historical_city=("city", join_unique),
        historical_county=("county", join_unique),
        historical_town=("town", join_unique),
        tools=("tool", join_unique),
    )
    .reset_index()
    .rename(
        columns={
            "city_o": "reported_current_city",
            "county_o": "reported_current_county",
        }
    )
)


# Attach the documented administrative change and its source.
reorganization_metadata = crosswalk.loc[
    ~crosswalk["same_area_for_qa"].eq("yes"),
    [
        "old_name",
        "current_name",
        "change_year",
        "change_type",
        "source_url",
    ],
].rename(
    columns={
        "old_name": "historical_county",
        "current_name": "reported_current_county",
        "source_url": "administrative_change_source_url",
    }
)

administrative_reorganization_review = (
    administrative_reorganization_review.merge(
        reorganization_metadata,
        on=[
            "historical_county",
            "reported_current_county",
        ],
        how="left",
        validate="many_to_one",
    )
)

# All administrative-reorganization cases are manually verified using the sources url retained below
# and the evidence documented in the Markdown section.
administrative_reorganization_review[
    "review_status"
] = "manually_verified_reorganization"

administrative_reorganization_review = (
    administrative_reorganization_review.sort_values(
        ["records", "lon"],
        ascending=[False, True],
    ).reset_index(drop=True)
)

display(administrative_reorganization_review)

,lon,lat,reported_current_city,reported_current_county,records,first_date,last_date,historical_city,historical_county,historical_town,tools,change_year,change_type,administrative_change_source_url,review_status
0,115.66400,28.73900,南昌市,新建区,170,2021-09-29,2025-10-21,南昌市,湾里区,招贤镇,烟炉,2019,district_abolished_and_merged,https://wl.nc.gov.cn/ncswlglj/gkgsgg/202509/b1...,manually_verified_reorganization
1,115.87600,29.42200,九江市,庐山市,45,2021-01-27,2025-09-09,九江市,星子县,温泉镇,火箭弹,2016,county_to_county_level_city,https://www.lushan.gov.cn/yxls/lsyg_194877/,manually_verified_reorganization
2,116.03600,29.44700,九江市,庐山市,24,2021-09-29,2023-08-07,九江市,星子县,南康镇,火箭弹,2016,county_to_county_level_city,https://www.lushan.gov.cn/yxls/lsyg_194877/,manually_verified_reorganization
3,116.06500,29.69400,九江市,濂溪区,16,2023-12-10,2025-09-10,九江市,庐山区,虞家河乡,火箭弹,2016,district_rename,https://www.xinhuanet.com/politics/2016-05/12/...,manually_verified_reorganization
4,116.03361,29.44722,九江,庐山市,6,2020-01-15,2020-12-29,九江市,星子县,南康镇,火箭,2016,county_to_county_level_city,https://www.lushan.gov.cn/yxls/lsyg_194877/,manually_verified_reorganization
5,115.94600,29.57000,九江市,濂溪区,5,2022-11-13,2022-11-14,九江市,庐山区,赛阳镇,火箭弹,2016,district_rename,https://www.xinhuanet.com/politics/2016-05/12/...,manually_verified_reorganization
6,116.06100,29.45100,九江市,庐山市,3,2023-07-29,2023-12-08,九江市,星子县,白鹿镇,火箭弹,2016,county_to_county_level_city,https://www.lushan.gov.cn/yxls/lsyg_194877/,manually_verified_reorganization
7,115.90100,29.34100,九江市,庐山市,1,2023-09-13,2023-09-13,九江市,星子县,横塘镇,火箭弹,2016,county_to_county_level_city,https://www.lushan.gov.cn/yxls/lsyg_194877/,manually_verified_reorganization
8,116.04600,29.51400,九江市,庐山市,1,2022-11-04,2022-11-04,九江市,星子县,白鹿镇,火箭弹,2016,county_to_county_level_city,https://www.lushan.gov.cn/yxls/lsyg_194877/,manually_verified_reorganization


### Interpretation

From the manual assessments, all 271 records are consistent with documented administrative reorganization, so the mismatches do not provide evidence of coordinate errors.

#### 湾里区 → 新建区: 170 records

All 170 records use `(115.664, 28.739)`, are dated 2021–2025, and fall in the shapefile's historical `湾里区–招贤镇` polygon. The operation files report `新建区`.

In 2019, the State Council approved abolishing 湾里区 and incorporating its administrative area into 新建区. Current government materials state that the former 湾里 towns and villages—including 招贤镇—are within the administrative adjustment while being administered by 湾里管理局. Therefore, the mismatch between reported `新建区` and coordinate-derived `湾里区–招贤镇` is consistent with the reorganization.

- [wiki page for 新建区](https://zh.wikipedia.org/wiki/%E6%96%B0%E5%BB%BA%E5%8C%BA)

#### 星子县 → 庐山市: 80 records

The 80 records span six coordinates in the historical `星子县` polygons: 温泉镇, 南康镇, 白鹿镇, and 横塘镇. They are dated 2020–2025 and report `庐山市`.

In 2016, 星子县 was abolished and the county-level 庐山市 was established. The new city includes the former 星子县 plus transferred territory from the former 庐山区. Because every reviewed coordinate lies within historical 星子县, the change from historical `星子县` to current `庐山市` is supported for these records.

- [wiki page for 庐山市](https://zh.wikipedia.org/wiki/%E5%BA%90%E5%B1%B1%E5%B8%82)

#### 庐山区 → 濂溪区: 21 records

The 21 records occur at two coordinates: 16 are assigned to historical `庐山区–虞家河乡` and 5 are assigned to historical `庐山区–赛阳镇`. They are dated 2022–2025 and report `濂溪区`.

In 2016, part of the former `庐山区` was transferred to the newly established `庐山市`. The territory that remained was renamed `濂溪区`. Therefore, the historical `庐山区` and current `濂溪区` boundaries are not identical. For our particular records, however, current official sources place both `虞家河乡` and `赛阳镇` in `濂溪区`, supporting the current reported district.

- [wiki page for 庐山市](https://zh.wikipedia.org/wiki/%E5%BA%90%E5%B1%B1%E5%B8%82)
- [wiki page for 九江市](https://zh.wikipedia.org/wiki/%E4%B9%9D%E6%B1%9F%E5%B8%82)
- [濂溪区 school zoning policy](https://jje.jiujiang.gov.cn/zwzx_194/gzdt/jyxw/202408/t20240814_6647889.html)

## 3.3 Cross-city observations

This section reviews records for which the reported city differs from the city assigned by the recorded coordinate. Repeated records are grouped by coordinate so the review focuses on 11 location cases rather than 143 individual operations.

The distance calculations use the same township shapefile as the cleaning pipeline. Coordinates and polygons are projected to `EPSG:32650` before distances are measured in meters. The calculations are diagnostics: proximity to a reported county, a nearest polygon, or a boundary does not establish the true operation location.

In [27]:
import geopandas as gpd


cross_city = mismatches.loc[
    mismatches["review_category"].eq("cross_city_review")
].copy()


def join_unique(values):
    values = [
        clean_name(value)
        for value in values
        if pd.notna(value) and clean_name(value)
    ]
    return " | ".join(dict.fromkeys(values))


cross_city_review = (
    cross_city.groupby(["lon", "lat"], dropna=False)
    .agg(
        records=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        reported_city=("city_o", join_unique),
        reported_county=("county_o", join_unique),
        coordinate_city=("city", join_unique),
        coordinate_county=("county", join_unique),
        coordinate_town=("town", join_unique),
        tools=("tool", join_unique),
    )
    .reset_index()
)

# The 2020 file contains a detailed reported site field that is not
# available in the later annual files.
operation_2020 = pd.read_excel(data_dir / "operation/2020.xlsx")
source_sites_2020 = (
    operation_2020.groupby(
        ["GPS经度", "GPS纬度"],
        dropna=False,
    )["作业地点"]
    .agg(join_unique)
    .reset_index()
    .rename(
        columns={
            "GPS经度": "lon",
            "GPS纬度": "lat",
            "作业地点": "source_site_2020",
        }
    )
)

cross_city_review = cross_city_review.merge(
    source_sites_2020,
    on=["lon", "lat"],
    how="left",
)

townships = gpd.read_file(
    data_dir / "township_shapefile/xiangzhen.shp"
)
townships = townships.loc[
    townships["省"].eq("江西省"),
    ["市", "县", "乡", "geometry"],
].copy()
townships_m = townships.to_crs("EPSG:32650")

townships_m["city_normalized"] = townships_m["市"].map(
    normalize_city
)
townships_m["county_normalized"] = townships_m["县"].map(
    normalize_county
)


def representative_name(value):
    return value.split(" | ")[0] if value else None


distance_results = []

for row in cross_city_review.itertuples(index=False):
    point_m = gpd.GeoSeries(
        [gpd.points_from_xy([row.lon], [row.lat])[0]],
        crs="EPSG:4326",
    ).to_crs("EPSG:32650").iloc[0]

    reported_city = representative_name(row.reported_city)
    reported_county = representative_name(row.reported_county)

    # The 2020 source combines the current and historical names in one field.
    if reported_county == "广信区（上饶县）":
        reported_county = "上饶县"

    reported_polygons = townships_m.loc[
        townships_m["city_normalized"].eq(
            normalize_city(reported_city)
        )
        & townships_m["county_normalized"].eq(
            normalize_county(reported_county)
        )
    ]

    coordinate_polygons = townships_m.loc[
        townships_m["市"].eq(row.coordinate_city)
        & townships_m["县"].eq(row.coordinate_county)
    ]

    reported_distance = (
        float(reported_polygons.geometry.distance(point_m).min())
        if len(reported_polygons)
        else pd.NA
    )
    coordinate_boundary_distance = (
        float(
            coordinate_polygons.geometry.union_all()
            .boundary.distance(point_m)
        )
        if len(coordinate_polygons)
        else pd.NA
    )

    distance_results.append(
        {
            "lon": row.lon,
            "lat": row.lat,
            "reported_city_county_distance_m": reported_distance,
            "coordinate_county_boundary_distance_m": (
                coordinate_boundary_distance
            ),
        }
    )

cross_city_review = cross_city_review.merge(
    pd.DataFrame(distance_results),
    on=["lon", "lat"],
    how="left",
)

suspected_coordinate_issue_coordinates = {
    (117.07528, 28.49028),
    (114.97000, 27.48000),
}

if cross_city_review["reported_city_county_distance_m"].isna().any():
    raise ValueError("Some cross-city cases are missing reported-location distances.")


def assign_cross_city_status(row):
    coordinate = (row["lon"], row["lat"])

    if coordinate in suspected_coordinate_issue_coordinates:
        return "suspected_coordinate_issue"

    if row["reported_city_county_distance_m"] <= 5_000:
        return "near_boundary_review"

    return "distant_location_conflict"


cross_city_review["review_status"] = cross_city_review.apply(
    assign_cross_city_status,
    axis=1,
)


cross_city_review[
    "reported_city_county_distance_m"
] = cross_city_review[
    "reported_city_county_distance_m"
].round(1)
cross_city_review[
    "coordinate_county_boundary_distance_m"
] = cross_city_review[
    "coordinate_county_boundary_distance_m"
].round(1)

cross_city_review = cross_city_review.sort_values(
    ["records", "lon"],
    ascending=[False, True],
).reset_index(drop=True)

cross_city_review.to_csv(
    data_dir / "check/cross_city_location_review.csv",
    index=False,
)

display(cross_city_review)

cross_city_status_summary = (
    cross_city_review.groupby("review_status")
    .agg(
        review_cases=("lon", "size"),
        records=("records", "sum"),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

cross_city_status_summary["share"] = (
    cross_city_status_summary["records"]
    / cross_city_status_summary["records"].sum()
).round(3)

display(cross_city_status_summary)

,lon,lat,records,first_date,last_date,reported_city,reported_county,coordinate_city,coordinate_county,coordinate_town,tools,source_site_2020,reported_city_county_distance_m,coordinate_county_boundary_distance_m,review_status
0,115.74600,27.11300,57,2021-01-06,2025-10-13,吉安市,永丰县,抚州市,乐安县,罗陂乡,烟炉,NaN,103.2,103.2,near_boundary_review
1,115.41800,27.56800,49,2021-02-09,2025-10-13,抚州市,乐安县,吉安市,峡江县,马埠镇,烟炉,NaN,17324.2,3593.8,distant_location_conflict
2,115.87200,27.85900,10,2023-07-15,2025-10-13,抚州市,崇仁县,宜春市,丰城市,洛市镇,烟炉,NaN,41.3,41.3,near_boundary_review
3,115.74583,27.11278,9,2020-01-15,2020-12-23,吉安,永丰,抚州市,乐安县,罗陂乡,碘化银地面发生器,陶唐高龙山360825010,95.1,95.1,near_boundary_review
4,117.27600,28.58200,7,2024-07-22,2025-08-06,鹰潭市,月湖区,上饶市,弋阳县,中畈乡,烟炉,NaN,35022.8,10.3,distant_location_conflict
5,114.97000,27.48000,4,2020-03-06,2020-06-24,南昌,安义县 | 安义,吉安市,吉水县,阜田镇,碘化银地面发生器,筲岭水库360123011 | 湾里区筲岭水库360123011,134041.0,2580.5,suspected_coordinate_issue
6,113.96000,27.01000,2,2022-10-07,2022-11-05,萍乡市,莲花县,吉安市,永新县,文竹镇,火箭弹,NaN,363.9,363.9,near_boundary_review
7,117.26000,28.61000,2,2021-12-14,2022-01-05,上饶市,弋阳县,鹰潭市,贵溪市,三县岭林场,烟炉,NaN,8.8,8.8,near_boundary_review
8,115.00600,29.07800,1,2022-10-08,2022-10-08,南昌市,新建区,九江市,武宁县,罗溪乡,火箭弹,NaN,74237.7,6148.4,distant_location_conflict
9,115.63700,29.44900,1,2022-08-29,2022-08-29,南昌市,新建区,九江市,德安县,吴山镇,火箭弹,NaN,48037.8,9190.7,distant_location_conflict


,review_status,review_cases,records,share
1,near_boundary_review,5,80,0.559
0,distant_location_conflict,4,58,0.406
2,suspected_coordinate_issue,2,5,0.035


### Notes on the cross-city review columns

Each row represents one coordinate, combining all operations recorded at that location.

- `reported_city` and `reported_county` preserve the source-file labels.
- `coordinate_city`, `coordinate_county`, and `coordinate_town` come from the township polygon containing the recorded coordinate.
- `reported_city_county_distance_m` is the shortest distance from the point to the reported city-county. A value of `0` means the point is inside or on those polygons; a small positive value indicates that it is just outside.
- `coordinate_county_boundary_distance_m` measures how far the point lies inside the coordinate-derived county from its boundary. It does not measure distance to the county as a whole.
- `source_site_2020` preserves the more detailed site name and administrative code available in the 2020 workbook.
- `review_status` records the concise manual review category used in the Markdown assessment below. It is an audit label, not a correction.
Distances are computed from the supplied township shapefile. They can identify boundary-sensitive and distant conflicts, but they cannot establish the true operation location.

### Interpretation

The 143 cross-city records reduce to 11 coordinate cases. The assessments below are manual review notes, not changes to the source data.

#### Near-boundary review: 80 records across 5 coordinates

For every case in this group, `reported_city_county_distance_m` and `coordinate_county_boundary_distance_m` are equal after rounding. This indicates that the reported and coordinate-derived county meet along the nearest mapped boundary. Because the two counties belong to different cities, this shared county boundary is also a city boundary. The recorded coordinates fall approximately 9–364 m to the coordinate-derived side of that boundary. . I also confirmed through online searching that all 5 pairs of city-county combination are adjacent. These small distances may reflect coordinate rounding differences or differences in the mapped boundary.

| Coordinates | Records | Reported → coordinate-derived location | Review note |
|---|---:|---|---|
| `(115.746, 27.113)` | 57 | 吉安市–永丰县 → 抚州市–乐安县–罗陂乡 | The distances to reported 吉安市–永丰县 and to the coordinate-derived 抚州市–乐安县 are both about 103 m. 
| `(115.74583, 27.11278)` | 9 | 吉安–永丰 → 抚州市–乐安县–罗陂乡 | The distances to reported 吉安市–永丰县 and to the coordinate-derived 抚州市–乐安县 boundary are both about 95 m. The 2020 source site is `陶唐高龙山360825010`--360825 is the adcode for the reported 吉安市–永丰县. |
| `(115.872, 27.859)` | 10 | 抚州市–崇仁县 → 宜春市–丰城市–洛市镇 | The distances to reported 抚州市–崇仁县 and to the coordinate-derived 宜春市–丰城市 boundary are both about 41 m. |
| `(113.960, 27.010)` | 2 | 萍乡市–莲花县 → 吉安市–永新县–文竹镇 | The distances to reported 萍乡市–莲花县 and to the coordinate-derived 吉安市–永新县 boundary are both about 364 m. |
| `(117.260, 28.610)` | 2 | 上饶市–弋阳县 → 鹰潭市–贵溪市–三县岭林场 | The distances to reported 上饶市–弋阳县 and to the coordinate-derived 鹰潭市–贵溪市 are both about 9 m. |

The 9 records from 2020 at `(115.74583, 27.11278)` identify the site as 陶唐高龙山360825010. The first six digits of the site code, 360825, correspond to the reported 永丰县, suggesting that the fixed site was administratively associated with 永丰县. The 57 records from 2021–2025 use the nearly identical coordinate (115.746, 27.113) and report the same county, 永丰县. This suggests that the 2021–2025 operations likely happen at the same fixed site as the 2020 records, although the later files do not include the detailed site codes. Both coordinates fall only about 95–103 m inside the adjacent 乐安县 polygon, so coordinate rounding or mapped-boundary differences are plausible explanations for the mismatch.

#### Distant location conflicts: 58 records across 4 coordinates

The 58 distant-conflict records occur at 4 coordinates located approximately 17–74 km from their reported counties, so the mismatches cannot plausibly be explained by coordinate rounding or boundary differences. Two coordinates account for 56 repeated smoke-furnace records, while the other two are isolated rocket records. The available data do not show whether the coordinates or reported locations are wrong.

| Coordinates | Records | Reported → coordinate-derived location | Review note |
|---|---:|---|---|
| `(115.418, 27.568)` | 49 | 抚州市–乐安县 → 吉安市–峡江县–马埠镇 | Repeated smoke-furnace records are about 17.3 km from reported 乐安 County. |
| `(117.276, 28.582)` | 7 | 鹰潭市–月湖区 → 上饶市–弋阳县–中畈乡 | Repeated smoke-furnace records are about 35 km from reported 月湖 District. |
| `(115.006, 29.078)` | 1 | 南昌市–新建区 → 九江市–武宁县–罗溪乡 | The rocket record is about 74 km from the reported 新建 District. 
| `(115.637, 29.449)` | 1 | 南昌市–新建区 → 九江市–德安县–吴山镇 | The rocket record is about 48 km from the reported 新建 District.

#### Suspected coordinate issues: 5 records across 2 coordinates

In this group, a coordinate problem is strongly suspected because their original records contians detailed site names--the first six digits in the site names are the administrative codes associated with the operation points: 360123 corresponds to 安义县, and 361121 corresponds to the former 上饶县, now 广信区. These codes agree with the reported locations, but the recorded coordinates are approximately 134 km and 61 km away from them. This provides strong evidence that the coordinates may contain coordinate entry errors.

| Coordinates | Records | Reported → coordinate-derived location | Review note |
|---|---:|---|---|
| `(114.970, 27.480)` | 4 | 南昌–安义县 → 吉安市–吉水县–阜田镇 | The 2020 source identifies `筲岭水库360123011` in the reported 安义 County, while the coordinate is about 134 km away. This strongly suggests a coordinate entry error. |
| `(117.07528, 28.49028)` | 1 | 上饶–广信区（上饶县） → 鹰潭市–余江县–画桥镇 | The 2020 source site `罗桥街道十里风荷361121017` supports the reported 上饶 County association, while the coordinate is about 61 km away. The intended coordinate remains unverified. |


## 3.4 Cross-county observations within the same city

This section reviews records for which the reported and coordinate-derived cities agree after formatting normalization, but the reported and coordinate-derived counties differ. The 616 records contain 49 unique coordinates and 55 coordinate–reported-county review cases. Records are grouped by coordinate and reported county so that the same coordinate remains separate when different records assign it different reported labels.

In [28]:
cross_county = mismatches.loc[
    mismatches["review_category"].eq("cross_county_same_city")
].copy()

cross_county_review = (
    cross_county.groupby(
        ["lon", "lat", "city_o", "county_o"],
        dropna=False,
    )
    .agg(
        records=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        coordinate_city=("city", join_unique),
        coordinate_county=("county", join_unique),
        coordinate_town=("town", join_unique),
        tools=("tool", join_unique),
    )
    .reset_index()
    .rename(
        columns={
            "city_o": "reported_city",
            "county_o": "reported_county",
        }
    )
)

cross_county_review = cross_county_review.merge(
    source_sites_2020,
    on=["lon", "lat"],
    how="left",
)

# Historical predecessors used only for distance calculations.
# These aliases do not make the reported and derived counties equivalent.
distance_reference_counties = {
    "濂溪区": "庐山区",
}

distance_results = []

for row in cross_county_review.itertuples(index=False):
    point_m = gpd.GeoSeries(
        [gpd.points_from_xy([row.lon], [row.lat])[0]],
        crs="EPSG:4326",
    ).to_crs("EPSG:32650").iloc[0]

    reported_county = clean_name(row.reported_county)

    distance_reference_county = distance_reference_counties.get(
        reported_county,
        reported_county,
    )


    reported_polygons = townships_m.loc[
        townships_m["city_normalized"].eq(
            normalize_city(row.reported_city)
        )
        & townships_m["county_normalized"].eq(
            normalize_county(distance_reference_county)
        )
    ]

    coordinate_polygons = townships_m.loc[
        townships_m["市"].eq(row.coordinate_city)
        & townships_m["县"].eq(row.coordinate_county)
    ]

    reported_distance = (
        float(reported_polygons.geometry.distance(point_m).min())
        if len(reported_polygons)
        else pd.NA
    )
    coordinate_boundary_distance = (
        float(
            coordinate_polygons.geometry.union_all()
            .boundary.distance(point_m)
        )
        if len(coordinate_polygons)
        else pd.NA
    )


    distance_results.append(
    {
        "lon": row.lon,
        "lat": row.lat,
        "reported_city": row.reported_city,
        "reported_county": row.reported_county,
        "distance_reference_county": distance_reference_county,
        "reported_county_distance_m": reported_distance,
        "coordinate_county_boundary_distance_m": (
            coordinate_boundary_distance
        ),
    }
)

cross_county_review = cross_county_review.merge(
    pd.DataFrame(distance_results),
    on=[
        "lon",
        "lat",
        "reported_city",
        "reported_county",
    ],
    how="left",
)


def assign_cross_county_status(row):
    distance = row["reported_county_distance_m"]

    if pd.isna(distance):
        return "reported_area_not_matched"

    if distance <= 5000:
        return "near_boundary_review"
    
    return "distant_location_conflict"


cross_county_review["review_status"] = (
    cross_county_review.apply(
        assign_cross_county_status,
        axis=1,
    )
)

for column in [
    "reported_county_distance_m",
    "coordinate_county_boundary_distance_m",
]:
    cross_county_review[column] = pd.to_numeric(
        cross_county_review[column],
        errors="coerce",
    ).round(1)

cross_county_review = cross_county_review.sort_values(
    ["review_status", "records", "lon"],
    ascending=[True, False, True],
).reset_index(drop=True)

cross_county_review.to_csv(
    data_dir / "check/cross_county_same_city_review.csv",
    index=False,
)

display(cross_county_review)

cross_county_status_summary = (
    cross_county_review.groupby("review_status")
    .agg(
        review_cases=("lon", "size"),
        records=("records", "sum"),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

cross_county_status_summary["share"] = (
    cross_county_status_summary["records"]
    / cross_county_status_summary["records"].sum()
).round(3)

display(cross_county_status_summary)

,lon,lat,reported_city,reported_county,records,first_date,last_date,coordinate_city,coordinate_county,coordinate_town,tools,source_site_2020,distance_reference_county,reported_county_distance_m,coordinate_county_boundary_distance_m,review_status
0,116.96300,28.09800,鹰潭市,月湖区,45,2021-01-25,2022-10-09,鹰潭市,贵溪市,龙虎山镇,烟炉,NaN,月湖区,8289.2,2464.9,distant_location_conflict
1,117.06800,28.44900,鹰潭市,月湖区,16,2023-01-22,2025-09-12,鹰潭市,贵溪市,塔桥园艺场,火箭弹,NaN,月湖区,16081.5,3060.6,distant_location_conflict
2,116.95900,28.09800,鹰潭市,月湖区,12,2022-10-27,2025-07-18,鹰潭市,贵溪市,龙虎山镇,烟炉,NaN,月湖区,8414.8,2536.8,distant_location_conflict
3,117.03300,27.98700,鹰潭市,月湖区,8,2024-07-28,2025-01-31,鹰潭市,贵溪市,上清镇,烟炉,NaN,月湖区,20676.2,736.5,distant_location_conflict
4,117.08000,28.44900,鹰潭市,月湖区,8,2022-01-05,2022-08-04,鹰潭市,贵溪市,塔桥园艺场,火箭弹,NaN,月湖区,16065.6,3010.8,distant_location_conflict
5,118.02600,28.13200,上饶市,信州区,6,2023-07-15,2023-07-23,上饶市,上饶县,五府山镇,火箭弹,NaN,信州区,27791.6,5583.4,distant_location_conflict
6,115.96100,28.24100,宜春市,上高县,5,2022-09-14,2022-11-16,宜春市,丰城市,白土镇,火箭弹,NaN,上高县,78524.3,4183.1,distant_location_conflict
7,117.16000,27.99200,鹰潭市,月湖区,5,2024-08-04,2025-07-08,鹰潭市,贵溪市,耳口乡,烟炉,NaN,月湖区,24403.3,9356.6,distant_location_conflict
8,116.94200,28.06900,鹰潭市,月湖区,3,2022-08-10,2024-08-06,鹰潭市,贵溪市,龙虎山镇,火箭弹,NaN,月湖区,12013.5,205.9,distant_location_conflict
9,117.53300,29.29200,上饶市,德兴市,3,2022-07-17,2022-10-09,上饶市,婺源县,赋春镇,火箭弹,NaN,德兴市,25400.3,7654.3,distant_location_conflict


,review_status,review_cases,records,share
1,near_boundary_review,16,437,0.718
0,distant_location_conflict,21,124,0.204
2,reported_area_not_matched,14,48,0.079


### Interpretation

#### Near-boundary review: 437 records across 16 cases

For 13 of the 15 cases in this group, `reported_county_distance_m` and `coordinate_county_boundary_distance_m` are equal after rounding. This indicates that the reported and coordinate-derived counties meet along the nearest mapped boundary, with the recorded coordinate falling approximately 41 m to 4.7 km on the coordinate-derived side. These relatively short distances likely reflect standardized site coordinates, coordinate rounding, or differences in the mapped boundary.

The two exceptions are `(113.860, 27.580)` and `(117.241, 29.310)`. Their distance measures differ by about 0.2 km and 0.7 km, respectively, because the nearest boundary of the coordinate-derived county is shared with a third county. The reported and coordinate-derived counties are still adjacent, but the distances refer to different boundary segments. Given the short distances and adjacency, coordinate rounding or differences in mapped boundaries is still very likely.

The largest/ most informative cases include:

- `(115.675, 28.792)`: 217 records report 南昌市–安义县 and are assigned to 南昌市–湾里区–太平镇. Both distances are about 1.3 km. The repeated fixed-site pattern makes a standardized/ rounded coordinate very likely.
- `(115.192, 29.492)`: 48 records report 九江市–武宁县 and are assigned to 九江市–瑞昌市–乐园乡. Both distances are about 171 m.
- `(115.014, 25.758)`: 33 records report 赣州市–赣县区 and are assigned to 赣州市–章贡区–沙石镇. Both distances are about 644 m.
- `(115.895, 29.704)`: 29 records report 九江市–濂溪区 but are assigned to historical 九江县–赛城湖水产场. Using 庐山区, the predecessor of 濂溪区, as the distance reference, both distances are about 1.37 km. 
- `(117.056, 28.299)`: 20 records report 鹰潭市–月湖区 and are assigned to 鹰潭市–贵溪市–江北街道. Both distances are about 278 m.
- `(115.523, 26.217)`: 8 records report 赣州市–于都县 and are assigned to 赣州市–兴国县–杰村乡. Both distances are about 87 m.
- `(115.95167, 27.56778)`: 7 records report 抚州–崇仁 and are assigned to 抚州市–乐安县–公溪镇. Both distances are about 71 m. The 2020 source site is `相山361024012`; `361024` is the administrative code for the reported 崇仁县, supporting the reported county association.


#### Distant location conflicts: 124 records across 21 cases

These coordinates are more than 5 km from the reported county, so ordinary coordinate rounding or map boundary differences are unlikely.

Several conflicts recur across multiple operations:

- `(116.963, 28.098)` and `(116.959, 28.098)` together account for 57 records reported as 月湖区 but assigned to 贵溪市–龙虎山镇. They are approximately 8.3–8.4 km from 月湖区.
- `(117.068, 28.449)`: 16 records reported as 月湖区 but assigned to 贵溪市–塔桥园艺场; approximately 16.1 km from 月湖区.
- `(117.033, 27.987)`: 8 records reported as 月湖区 but assigned to 贵溪市–上清镇; approximately 20.7 km from 月湖区.
- `(117.160, 27.992)`: 5 records reported as 月湖区 but assigned to 贵溪市–耳口乡; approximately 24.4 km from 月湖区.
- `(118.026, 28.132)`: 6 records reported as 信州区 but assigned to 上饶县–五府山镇; approximately 27.8 km from 信州区.
- `(115.961, 28.241)`: 5 records reported as 上高县 but assigned to 丰城市–白土镇; approximately 78.5 km from 上高县.

Among them, the most systematic pattern involves 月湖区 and 贵溪市. Across 99 records at nine coordinates, the source reports 月湖区 while the coordinates are assigned to 贵溪市, approximately 8–24 km from 月湖区. Although these counties are adjacent, their far distance makes rounding error less plausible. This recurring pattern may reflect a systematic reporting convention, administrative responsibility for operations outside 月湖区, or repeated source-label errors(although unlikely); the available records do not distinguish among these explanations.

Some 2020 records contain site names and administrative codes that support the reported counties. This provides stronger evidence that the reported counties were intended and that the coordinates may contain data entry errors:

- `(115.69278, 26.80639)` reports 兴国 and records site name of`方山岭360732013`, but the coordinate is assigned to 宁都县 and lies about 14.5 km from 兴国县. The site code begins with `360732`, the administrative code for the reported 兴国县, supporting the reported county association.
- `(114.67472, 26.61750)` reports 遂川 and records `佛祖仙岭360827009`, but the coordinate is assigned to 万安县 and lies about 10.6 km from 遂川县.  The site code begins with `360827`, the adcode for the reported 遂州县.
- `(114.18306, 26.40000)` reports 永新 and records `临时360830010`, but the coordinate is assigned to 井冈山市 and lies about 36 km from 永新县. The site code begins with `360830`, the adcode for the reported 永新县.
- `(115.25583, 26.83583)` reports 永新 and records `万年山360830100`, but the coordinate is assigned to 青原区 and lies about 77 km from 永新县. The site code begins with `360830`, the adcode for the reported 永新县.


#### Reported areas not matched to county polygons: 48 records across 14 cases

The remaining reported-area labels cannot be matched directly to ordinary county polygons in the shapefile. A missing `reported_county_distance_m` means that no corresponding county polygon was found.

These labels fall into 3 groups:

##### Invalid reported value with the recurring 月湖区/贵溪市 mismatch: 14 identical records
Fourteen 2021 records use `中华人民共和国` at `(117.080, 28.449)`. `中华人民共和国` is not a county-level location, it is certainly a county entry error. The coordinate is assigned to 贵溪市–塔桥园艺场, however, eight 2022 records at the same coordinate report 月湖区. This reproduces the recurring 月湖区/贵溪市 disagreement we identified above.

##### Generic labels: 14 records across 8 cases

Several 2020 records use generic county labels(broad city-level names) such as `市区`, `城区`, `鹰潭`, or `新余市本级`, but their detailed site names support the coordinate-derived counties:

- **宜春–市区:** `南庙镇梅花村360902013` supports 袁州区.
- **赣州–市区:** `峰山360702015` supports 章贡区.
- **上饶–市区:** `茅家岭乡361102004` supports 信州区.
- **萍乡–城区:** `城东飞碟厂360302002` supports 安源区.
- **新余市本级:** `下村镇狮子口水库坝下360502007` supports 渝水区.

##### Generic labels with the recurring 月湖区/贵溪市 mismatch: 16 records across 3 cases

- **塔桥:** Six records use `鹰潭` or `市区` and identify the site as `塔桥360602006`. Their coordinates are assigned to 鹰潭市-贵溪市–塔桥园艺场, so the chinese character in the site name directly supports the coordinate-derived location. However, the code in the site name begins with `360602`, which corresponds to 鹰潭市-月湖区.
- **天门山:** Ten records use `鹰潭–市区` and identify the site as `天门山360602008`. The site name and coordinates support 贵溪市–上清镇, while the `360602` prefix again corresponds to 月湖区.

In both groups, the site name and coordinate-derived location support 贵溪市, while the embedded ad code supports 月湖区. This reproduces the recurring 月湖区/贵溪市 disagreement and likely reflects a systematic reporting or site-coding convention.